In [ ]:
import pandas as pd
from utils.notebook_helpers import get_postgres_engine

engine, pg_conf = get_postgres_engine("../configs/database.yaml")
print("Connected to PostgreSQL:", pg_conf["host"], pg_conf["db"])

Connected to PostgreSQL: localhost storage_analytics


In [2]:
curated_df = pd.read_sql_query("SELECT * FROM curated_device_metrics", engine)
anomaly_df = pd.read_sql_query("SELECT * FROM anomaly_events", engine)

curated_df["timestamp"] = pd.to_datetime(curated_df["timestamp"])
anomaly_df["timestamp"] = pd.to_datetime(anomaly_df["timestamp"])

In [3]:
curated_df.groupby("workload_pattern")[[
    "total_iops",
    "avg_latency_ms",
    "saturation_score",
    "merge_efficiency",
    "queue_efficiency",
    "iowait_pressure",
    "svctm_await_ratio",
    "write_amplification",
]].mean().round(4).sort_values("avg_latency_ms", ascending=False)

,total_iops,avg_latency_ms,saturation_score,merge_efficiency,queue_efficiency,iowait_pressure,svctm_await_ratio,write_amplification
workload_pattern,,,,,,,,
latency_sensitive,326.3228,240.5644,123.8725,0.1454,617.4450,11.7132,1.2143,1.6105
saturated,684.2864,2.8966,135.9627,0.0826,821.2985,12.2512,5.5579,1.1896
small_io_pressure,725.0815,2.3171,10.1477,0.0126,5576.9049,1.8772,0.8554,0.6037
balanced,454.7691,1.2286,22.1950,0.0540,1331.0221,0.9813,0.7011,0.8333
burst_io,1609.8864,0.9357,59.7635,0.0250,2689.3794,3.1019,0.3624,0.6180


In [4]:
anomaly_df.groupby("workload_pattern").size().sort_values(ascending=False)

workload_pattern
latency_sensitive    950
balanced             871
burst_io             672
saturated            297
small_io_pressure    100
dtype: int64

In [5]:
anomaly_df["root_cause_hint"].value_counts()

root_cause_hint
saturation_score deviated from device baseline                702
iowait_pressure deviated from device baseline                 530
avg_latency_ms deviated from device baseline                  449
queue_efficiency deviated from device baseline                434
total_iops deviated from device baseline                      277
merge_efficiency deviated from device baseline                267
Device saturation with queue buildup                           99
total_throughput_mb_s deviated from device baseline            75
CPU I/O wait pressure indicates backend storage bottleneck     44
Small random I/O pressure reducing merge effectiveness         13
Name: count, dtype: int64

In [6]:
anomaly_df[anomaly_df["severity"] == "critical"][
    ["device", "timestamp", "metric_name", "severity", "workload_pattern", "root_cause_hint"]
].sort_values("timestamp")

,device,timestamp,metric_name,severity,workload_pattern,root_cause_hint
262,dm-0,2026-03-30 00:32:26.521293+00:00,queue_efficiency,critical,balanced,queue_efficiency deviated from device baseline
263,dm-0,2026-03-30 01:22:19.837735+00:00,queue_efficiency,critical,balanced,queue_efficiency deviated from device baseline
268,dm-0,2026-03-30 07:17:40.150387+00:00,queue_efficiency,critical,balanced,queue_efficiency deviated from device baseline
2169,sdb,2026-03-30 23:07:32.330347+00:00,avg_latency_ms,critical,latency_sensitive,avg_latency_ms deviated from device baseline
1068,nvme1n1,2026-03-31 09:07:52.214182+00:00,avg_latency_ms,critical,latency_sensitive,avg_latency_ms deviated from device baseline
...,...,...,...,...,...,...
234,dm-0,2026-04-05 18:07:11.720363+00:00,saturation_score,critical,latency_sensitive,Device saturation with queue buildup
475,dm-0,2026-04-05 18:07:11.720363+00:00,iowait_pressure,critical,latency_sensitive,CPU I/O wait pressure indicates backend storag...
476,dm-0,2026-04-05 18:12:40.108486+00:00,iowait_pressure,critical,latency_sensitive,iowait_pressure deviated from device baseline
56,dm-0,2026-04-05 18:12:40.108486+00:00,avg_latency_ms,critical,latency_sensitive,avg_latency_ms deviated from device baseline


In [7]:
anomaly_df.groupby(["device", "root_cause_hint"]).size().sort_values(ascending=False).head(20)

device   root_cause_hint                               
sdb      avg_latency_ms deviated from device baseline      253
nvme0n1  iowait_pressure deviated from device baseline     179
         saturation_score deviated from device baseline    176
dm-0     saturation_score deviated from device baseline    174
nvme1n1  saturation_score deviated from device baseline    155
sdb      queue_efficiency deviated from device baseline    145
sda      iowait_pressure deviated from device baseline     142
dm-0     queue_efficiency deviated from device baseline    111
nvme1n1  iowait_pressure deviated from device baseline     110
sda      saturation_score deviated from device baseline    107
nvme1n1  queue_efficiency deviated from device baseline    102
dm-0     iowait_pressure deviated from device baseline      99
nvme0n1  merge_efficiency deviated from device baseline     98
sdb      saturation_score deviated from device baseline     90
         Device saturation with queue buildup               89

In [8]:
anomaly_df.groupby(["metric_name", "root_cause_hint"]).size().sort_values(ascending=False).head(25)

metric_name            root_cause_hint                                           
saturation_score       saturation_score deviated from device baseline                702
iowait_pressure        iowait_pressure deviated from device baseline                 530
avg_latency_ms         avg_latency_ms deviated from device baseline                  449
queue_efficiency       queue_efficiency deviated from device baseline                434
total_iops             total_iops deviated from device baseline                      277
merge_efficiency       merge_efficiency deviated from device baseline                267
saturation_score       Device saturation with queue buildup                           82
total_throughput_mb_s  total_throughput_mb_s deviated from device baseline            75
iowait_pressure        CPU I/O wait pressure indicates backend storage bottleneck     44
avg_latency_ms         Device saturation with queue buildup                           17
                       Small